In [ ]:
import numpy as np

class Layer:
    def __init__(self, input_size, n_units):
        self.input_size = input_size
        self.n_units = n_units
        self.input = None
        self.z = None
        self.output = None
        
        # Initialize weights and biases
        self.weights = None
        self.biases = np.zeros((1, n_units))
        self.dWeights = None
        self.dBiases = None
        
    def linear_forward(self, X):
        return np.dot(X, self.weights) + self.biases
    
    def calc_gradients(self, dz):
        self.dWeights = np.dot(self.input.T, dz)
        self.dBiases = np.sum(dz, axis=0, keepdims=True)
        
    def forward(self, X):
        self.input = X
        z = self.linear_forward(X)
        self.output = self.activation_func(z)
        return self.output
    
    def backward(self, dLoss):
        dz = self.activation_grad(dLoss)
        self.calc_gradients(dz)
        dX = np.dot(dz, self.weights.T)
        return dX
    
    def activation_func(self, z):
        pass
    
    def activation_grad(self, dLoss):
        pass
    
    def update_params(self, learning_rate):
        self.weights -= learning_rate * self.dWeights
        self.biases -= learning_rate * self.dBiases
    
class ReLULayer(Layer):
    def __init__(self, input_size, n_units):
        super().__init__(input_size, n_units)
        self.weights = np.random.randn(input_size, n_units) * np.sqrt(2. / input_size)
        
    def activation_func(self, z):
        return np.maximum(0, z)
    
    def activation_grad(self, dLoss):
        dz = dLoss.copy()
        dz[self.output <= 0] = 0
        return dz
    
class SigmoidLayer(Layer):
    def __init__(self, input_size, n_units):
        super().__init__(input_size, n_units)
        self.weights = np.random.randn(input_size, n_units) * np.sqrt(1. / input_size)
        
    def activation_func(self, z):
        return 1 / (1 + np.exp(-z))
    
    def activation_grad(self, dLoss):
        dz = dLoss * (self.output * (1 - self.output))
        return dz
    
class TanhLayer(Layer):
    def __init__(self, input_size, n_units):
        super().__init__(input_size, n_units)
        self.weights = np.random.randn(input_size, n_units) * np.sqrt(1. / input_size)
        
    def activation_func(self, z):
        return np.tanh(z)
    
    def activation_grad(self, dLoss):
        dz = dLoss * (1 - self.output ** 2)
        return dz

class NuralNetwork:
    def __init__(self, input_size=None):
        self.layers = []
        self.next_input_size = input_size
    
    def add_layer(self, n_units, activation):
        match activation:
            case 'relu':
                self.layers.append(ReLULayer(self.next_input_size, n_units))
            case 'sigmoid':
                self.layers.append(SigmoidLayer(self.next_input_size, n_units))
            case 'tanh':
                self.layers.append(TanhLayer(self.next_input_size, n_units))
            case _:
                raise ValueError(f"Unknown activation function: {activation}")
        self.next_input_size = n_units
        
    def forward(self, X):
        output = X
        for layer in self.layers:
            output = layer.forward(output)
        return output
    
    def backward(self, dLoss):
        grad = dLoss
        for layer in reversed(self.layers):
            grad = layer.backward(grad)
        return grad
    
    def update_params(self, learning_rate):
        for layer in self.layers:
            layer.update_params(learning_rate)
            
    def fit(self, X, y, learning_rate=0.01, epochs=1000):
        for epoch in range(epochs):
            # Forward propagation
            y_pred = self.forward(X)
            loss = np.mean((y_pred - y) ** 2)
            if (epoch + 1) % 100 == 0:
                print(f"Epoch {epoch+1}/{epochs}, Loss: {loss}")
                
            # Backward propagation
            dLoss = 2 * (y_pred - y) / len(y)
            self.backward(dLoss)
            self.update_params(learning_rate)
            
    def predict(self, X):
        return self.forward(X)


In [120]:
model = NuralNetwork(input_size=3)
model.add_layer(n_units=5, activation='relu')
model.add_layer(n_units=2, activation='tanh')
model.add_layer(n_units=1, activation='sigmoid')
X = np.array([[0.1, 0.2, 0.3],
              [0.4, 0.5, 0.6],
              [0.7, 0.8, 0.9]])
y = np.array([[1], [1], [0]])
model.fit(X, y, learning_rate=0.04, epochs=2000)
predictions = model.predict(X)
labels = predictions.copy()
for i, output in enumerate(labels):
    if output > 0.5:
        labels[i] = 1
    else:
        labels[i] = 0
print(f"Predictions:{predictions}")
print(f"labels:{labels}")

Epoch 100/2000, Loss: 0.1672930789209169
Epoch 200/2000, Loss: 0.11765207321267011
Epoch 300/2000, Loss: 0.08286031339683349
Epoch 400/2000, Loss: 0.058791250158734466
Epoch 500/2000, Loss: 0.04171677968497902
Epoch 600/2000, Loss: 0.030048521333326434
Epoch 700/2000, Loss: 0.02224976779706798
Epoch 800/2000, Loss: 0.017011899469212898
Epoch 900/2000, Loss: 0.013416063987361046
Epoch 1000/2000, Loss: 0.010875393396485281
Epoch 1100/2000, Loss: 0.00902610670250448
Epoch 1200/2000, Loss: 0.007641943293740082
Epoch 1300/2000, Loss: 0.006579524452225365
Epoch 1400/2000, Loss: 0.005745722044946107
Epoch 1500/2000, Loss: 0.0050784433636566545
Epoch 1600/2000, Loss: 0.004535218268347036
Epoch 1700/2000, Loss: 0.004086293791609558
Epoch 1800/2000, Loss: 0.003710361137945184
Epoch 1900/2000, Loss: 0.0033918487818198994
Epoch 2000/2000, Loss: 0.0031191676344274437
Predictions:[[0.98265113]
 [0.93427231]
 [0.06876634]]
labels:[[1.]
 [1.]
 [0.]]
